In [3]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
H    = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz 200 produktów z opisami, przekrojowo po kategoriach
q = '''*[_type=="product" && defined(shortDescription) && string::length(shortDescription) > 20][0..199] | order(_updatedAt desc) {
  _id, name, sku,
  "brandName": brand->name,
  "catName": category->name,
  "catSlug": category->slug.current,
  "catL1": category->parent->parent->name,
  shortDescription,
  description,
  "specCount": count(technicalSpec),
  technicalSpec
}'''

products = requests.get(BASE, params={"query": q}, headers=H).json()["result"]
print(f"Pobrano: {len(products)} produktów")

# Podsumowanie strukturalne
no_desc       = [p for p in products if not p.get("description")]
no_short      = [p for p in products if not p.get("shortDescription")]
no_specs      = [p for p in products if (p.get("specCount") or 0) == 0]
short_desc    = [p for p in products if p.get("shortDescription") and len(p["shortDescription"]) < 50]
generic_terms = ["materiał", "produkt", "artykuł", "towar", "rzecz", "item"]

print(f"\n=== STATYSTYKI STRUKTURALNE ===")
print(f"Brak description (długi):  {len(no_desc)}/{len(products)}")
print(f"Brak shortDescription:     {len(no_short)}/{len(products)}")
print(f"Brak technicalSpec:        {len(no_specs)}/{len(products)}")
print(f"shortDesc < 50 znaków:     {len(short_desc)}/{len(products)}")

# Zapisz do dalszej analizy
import json
with open("/tmp/products_sample.json", "w") as f:
    json.dump(products, f, ensure_ascii=False)

print("\nGotowe — dane do analizy jakości")


Pobrano: 200 produktów

=== STATYSTYKI STRUKTURALNE ===
Brak description (długi):  41/200
Brak shortDescription:     0/200
Brak technicalSpec:        41/200
shortDesc < 50 znaków:     0/200

Gotowe — dane do analizy jakości


In [7]:

import json, re
from collections import Counter, defaultdict

with open("/tmp/products_sample.json") as f:
    products = json.load(f)

# ═══ ANALIZA 1: Błędy terminologiczne ═══
TERM_ERRORS = {
    # Błędne uproszczenia
    "do izolacji termicznej": "zamiast: 'termoizolacyjny'",
    "do ocieplenia": "OK — ale sprawdź kontekst",
    "cement portlandzki": "sprawdź poprawność dla produktu",
    "schnięcia": "OK",
    "suche powietrze": "podejrzane — zwykle 'w temp. pokojowej'",
    # Zbyt ogólne
    "stosowany w budownictwie": "ZA OGÓLNE",
    "produkt wysokiej jakości": "ZA OGÓLNE / MARKETINGOWE",
    "doskonały do": "MARKETINGOWE",
    "idealny do": "MARKETINGOWE",
    "najlepszy": "MARKETINGOWE / NIEPRAWDZIWE",
    "profesjonalny produkt": "ZA OGÓLNE",
    "szeroko stosowany": "ZA OGÓLNE",
}

WATCH_PHRASES = [
    ("idealny do", "marketingowe, unikać"),
    ("doskonały do", "marketingowe, unikać"),
    ("najwyższej jakości", "pusty slogan"),
    ("produkt wysokiej jakości", "pusty slogan"),
    ("szeroko stosowany", "zbyt ogólne"),
    ("stosowany w budownictwie", "zbyt ogólne"),
    ("łatwy w użyciu", "zbyt ogólne bez kontekstu"),
    ("gwarantuje", "zbyt mocne twierdzenie"),
    ("materiał budowlany", "zbyt ogólne jako cała charakterystyka"),
]

watch_hits = defaultdict(list)
for p in products:
    text = (p.get("shortDescription","") + " " + (p.get("description") or "")).lower()
    for phrase, reason in WATCH_PHRASES:
        if phrase in text:
            watch_hits[phrase].append(p["name"][:50])

print("=== PROBLEMATYCZNE FRAZY ===")
for phrase, products_list in sorted(watch_hits.items(), key=lambda x: -len(x[1])):
    print(f"  '{phrase}' ({len(products_list)}x) — {dict(WATCH_PHRASES)[phrase]}")
    for n in products_list[:2]:
        print(f"    np. {n}")

# ═══ ANALIZA 2: Spójność technicalSpec ═══
print("\n=== ANALIZA TECHNICALSPEC ===")
all_labels = Counter()
missing_labels = Counter()
wrong_values = []

for p in products:
    specs = p.get("technicalSpec") or []
    labels = [s.get("label","") for s in specs]
    all_labels.update(labels)
    
    # Sprawdź czy wartości nie są placeholderami
    for s in specs:
        val = s.get("value","")
        if val in ["...", "—", "-", "N/A", "brak danych", "zależy od produktu", ""]:
            wrong_values.append(f"{p['name'][:40]} → {s['label']}: '{val}'")
        # Sprawdź czy wartość nie jest taka sama jak etykieta
        if val.lower() == s.get("label","").lower():
            wrong_values.append(f"{p['name'][:40]} → {s['label']} = wartość identyczna z etykietą")

print(f"Top etykiety specs:")
for label, cnt in all_labels.most_common(10):
    print(f"  '{label}': {cnt}x")

print(f"\nBłędne/placeholder wartości ({len(wrong_values)} szt.):")
for w in wrong_values[:10]:
    print(f"  {w}")

# ═══ ANALIZA 3: Kategorie vs treść — spójność ═══
print("\n=== SPÓJNOŚĆ KATEGORIA vs OPIS ===")
mismatches = []
for p in products:
    cat  = (p.get("catSlug") or "").lower()
    text = (p.get("shortDescription","") + " " + (p.get("description") or "")).lower()
    name = p.get("name","").lower()
    
    # Sprawdź czy opis pasuje do kategorii
    issues = []
    if "farb" in cat and "farb" not in text and "emal" not in text and "lakier" not in text:
        issues.append("kat=farby, brak słowa 'farba/emalia/lakier' w opisie")
    if "tynk" in cat and "tynk" not in text:
        issues.append("kat=tynki, brak 'tynk' w opisie")
    if "styropian" in cat and "styropian" not in text and "eps" not in text:
        issues.append("kat=styropian, brak 'styropian/EPS' w opisie")
    if "klej" in cat and "klej" not in text and "adhez" not in text:
        issues.append("kat=kleje, brak 'klej' w opisie")
    
    if issues:
        mismatches.append((p["name"][:50], p.get("catSlug",""), issues))

print(f"Niezgodności kategoria↔opis: {len(mismatches)}")
for name, cat, issues in mismatches[:8]:
    print(f"  [{cat}] {name}")
    for i in issues: print(f"    ⚠ {i}")

# ═══ ANALIZA 4: Próbki — szczegółowe ═══
print("\n=== PRÓBKI OPISÓW — OCENA SZCZEGÓŁOWA ===")
# Losuj po jednym z 5 kategorii
by_cat = defaultdict(list)
for p in products:
    by_cat[p.get("catL1","inne")].append(p)

shown = 0
for cat, prods in sorted(by_cat.items())[:6]:
    p = prods[0]
    print(f"\n[{cat} / {p.get('catName','')}]")
    print(f"  NAZWA: {p['name']}")
    print(f"  SHORT: {p.get('shortDescription','')[:200]}")
    desc = p.get("description") or ""
    print(f"  DESC:  {desc[:200] if desc else '— BRAK —'}")
    specs = p.get("technicalSpec") or []
    for s in specs[:3]:
        print(f"  SPEC:  {s.get('label','')} = {s.get('value','')}")
    shown += 1


=== PROBLEMATYCZNE FRAZY ===
  'gwarantuje' (77x) — zbyt mocne twierdzenie
    np. Lepik Do Izo. P Abizol 9Kg
    np. Everal Aqua 10 \
  'idealny do' (35x) — marketingowe, unikać
    np. Akr. Szar J.Sat 5L
    np. Pigment Szafir D14 Sentic
  'materiał budowlany' (20x) — zbyt ogólne jako cała charakterystyka
    np. Bloczek Ytong Solid Pp5/0,6 S+Gt 24Cm 48Szt/Pal
    np. Silka Blok Eq1224 Kl.20
  'łatwy w użyciu' (1x) — zbyt ogólne bez kontekstu
    np. Dodatek Do Klejów Top Latex Eco 8Kg

=== ANALIZA TECHNICALSPEC ===
Top etykiety specs:
  'Typ produktu': 159x
  'Zastosowanie': 159x
  'Opakowanie': 157x
  'Temperatura aplikacji': 152x
  'Wydajność': 151x
  'Materiał': 6x
  'Średnica': 4x
  'Marka': 3x
  'Gęstość': 2x
  'Wymiary': 2x

Błędne/placeholder wartości (0 szt.):

=== SPÓJNOŚĆ KATEGORIA vs OPIS ===
Niezgodności kategoria↔opis: 22
  [farby-do-drewna] Olej Do Tarasu 2,5L Naturalny
    ⚠ kat=farby, brak słowa 'farba/emalia/lakier' w opisie
  [farby-do-drewna] Olej Superdeck Do Tar

TypeError: '<' not supported between instances of 'NoneType' and 'str'

In [11]:

import json, re
from collections import Counter, defaultdict

with open("/tmp/products_sample.json") as f:
    products = json.load(f)

# Próbki per kategoria L1
by_cat = defaultdict(list)
for p in products:
    key = p.get("catL1") or "inne"
    by_cat[key].append(p)

print("=== PRÓBKI OPISÓW PER KATEGORIA ===\n")
for cat in sorted(by_cat.keys(), key=lambda x: (x is None, x))[:8]:
    p = by_cat[cat][0]
    short = p.get("shortDescription","") or ""
    desc  = p.get("description","")  or ""
    specs = p.get("technicalSpec")   or []
    print(f"[L1: {cat} | kat: {p.get('catName','')}]")
    print(f"  NAZWA: {p['name']}")
    print(f"  SHORT: {short[:180]}")
    print(f"  DESC:  {desc[:180] if desc else '— BRAK DŁUGIEGO OPISU —'}")
    for s in specs[:3]:
        print(f"  SPEC:  {s.get('label','')} = {s.get('value','')}")
    print()

# Dodatkowe — produkty z "..." w specach lub za krótkim opisem
print("=== PODEJRZANE WARTOŚCI W SPECS ===")
placeholder_vals = ["...", "—", "zależy od produktu", "wg producenta", "patrz etykietę", "N/A"]
suspect = []
for p in products:
    for s in (p.get("technicalSpec") or []):
        v = s.get("value","")
        if any(v.startswith(ph) for ph in ["zob", "wg ", "patrz", "zgodnie"]) or v in placeholder_vals or len(v) < 2:
            suspect.append(f"[{p['name'][:40]}] {s['label']}: '{v}'")
if suspect:
    for s in suspect[:10]: print(f"  {s}")
else:
    print("  Brak ewidentnych placeholderów ✅")

# Duplikaty opisów — czy AI copy-pastuje ten sam opis dla różnych produktów
print("\n=== DUPLIKATY shortDescription ===")
desc_count = Counter(p.get("shortDescription","") for p in products)
dupes = [(d, c) for d, c in desc_count.items() if c > 1 and len(d) > 30]
print(f"Identyczne shortDesc: {len(dupes)}")
for desc, cnt in sorted(dupes, key=lambda x: -x[1])[:5]:
    print(f"  ({cnt}x) {desc[:100]}")

# Długości
shorts = [len(p.get("shortDescription","")) for p in products if p.get("shortDescription")]
descs  = [len(p.get("description",""))      for p in products if p.get("description")]
print(f"\n=== DŁUGOŚCI ===")
print(f"shortDesc: min={min(shorts)} avg={sum(shorts)//len(shorts)} max={max(shorts)} | n={len(shorts)}")
if descs:
    print(f"description: min={min(descs)} avg={sum(descs)//len(descs)} max={max(descs)} | n={len(descs)}")
else:
    print("description: brak danych")


=== PRÓBKI OPISÓW PER KATEGORIA ===

[L1: Chemia budowlana | kat: Kleje do styropianu i styroduru]
  NAZWA: Klej Do Siat.25Kg Grip U
  SHORT: Atlas Klej do siatki 25kg Grip U to wysokiej jakości zaprawa klejąca przeznaczona do przyklejania płyt termoizolacyjnych ze styropianu oraz zatapiania siatki z włókna szklanego w s
  DESC:  Klej do siatki 25kg Grip U marki Atlas to nowoczesna zaprawa klejąca opracowana z myślą o systemach izolacji termicznej budynków. Jej unikalna formuła zapewnia doskonałą przyczepno
  SPEC:  Typ produktu = Zaprawa klejąca do styropianu i zatapiania siatki
  SPEC:  Zastosowanie = Przyklejanie płyt termoizolacyjnych ze styropianu, zatapianie siatki zbrojącej w systemach ociepleń
  SPEC:  Opakowanie = Worek 25 kg

[L1: Dachy | kat: Okna dachowe]
  NAZWA: Okno Dachowe Fakro Ptp-V U4 66X118
  SHORT: Okno dachowe Fakro Ptp-V U4 to innowacyjne rozwiązanie zapewniające naturalne światło i wentylację na poddaszu. Charakteryzuje się doskonałą izolacyjnością termiczną dzi

In [15]:

import requests, json, os

SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT_ID = "nzcwegq7"
DATASET = "production"

def groq(query, params=None):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    r = requests.get(url, headers={"Authorization": f"Bearer {SANITY_TOKEN}"},
                     params={"query": query, **(params or {})})
    r.raise_for_status()
    return r.json()["result"]

# Łączna liczba produktów
total = groq('count(*[_type == "product"])')
print(f"Łącznie produktów: {total}")

# Ile ma shortDescription
with_desc = groq('count(*[_type == "product" && defined(shortDescription) && shortDescription != ""])')
print(f"Ma shortDescription: {with_desc}")

# Ile ma description (długi)
with_long = groq('count(*[_type == "product" && defined(description) && description != ""])')
print(f"Ma description (długi): {with_long}")

# Ostatni log
log_path = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log2.txt"
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f"\nOstatnie 5 linii logu:")
    for l in lines[-5:]:
        print(l.rstrip())


Łącznie produktów: 15921


Ma shortDescription: 2522


Ma description (długi): 2401

Ostatnie 5 linii logu:
[2026-05-29 15:52:37]   LLM sparsowanych: 423 / 466
[2026-05-29 15:52:40]   PATCH: 423 dokumentów zapisanych
[2026-05-29 15:52:40]   Postęp: updated=2152, skip=243, err=105
[2026-05-29 15:52:41] Offset 2500: pobrano 500 produktów
[2026-05-29 15:52:41]   Po filtracji: 457 dobrych, 43 pominiętych


In [19]:

# ============================================================
# PIPELINE V4 — SEO names + opisy bez marketingu
# Wszystkie 15921 produktów od offset 0
# ============================================================

import sys, json, re, time, requests, asyncio, logging
from datetime import datetime

sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

# ——— Konfiguracja ————————————————————————————————————————————
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT_ID  = "nzcwegq7"
DATASET     = "production"
BATCH_SIZE  = 500          # produktów na transzę
LLM_CHUNK   = 100          # batch_llm chunk
PATCH_CHUNK = 50           # PATCH do Sanity
LOG_FILE    = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log2.txt"
START_OFFSET = 0           # restart od 0 — SEO names dla wszystkich

# ——— Helpers ————————————————————————————————————————————————
def log(msg):
    ts  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

def groq(query, params=None):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    r = requests.get(url,
                     headers={"Authorization": f"Bearer {SANITY_TOKEN}"},
                     params={"query": query, **(params or {})},
                     timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def sanity_patch(mutations):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/mutate/{DATASET}"
    r = requests.post(url,
                      headers={"Authorization": f"Bearer {SANITY_TOKEN}",
                               "Content-Type": "application/json"},
                      json={"mutations": mutations},
                      timeout=60)
    r.raise_for_status()
    return r.json()

# ——— Filtry nazw ————————————————————————————————————————————
BAD_NAME_RE = re.compile(
    r'^P-\d+$'                          # SKU-only P-0000123
    r'|^[A-Z0-9\-]{1,8}$'              # krótki kod bez spacji
    r'|^[A-Z]{4,}$',                    # same wielkie litery
    re.IGNORECASE
)

def is_bad_name(name: str) -> bool:
    if not name or len(name) < 4:
        return True
    caps = sum(1 for c in name if c.isupper())
    if caps / max(len(name), 1) > 0.7:
        return True
    if ' ' not in name and len(name) < 10:
        return True
    return bool(BAD_NAME_RE.match(name.strip()))

def is_bad_brand(brand: str) -> bool:
    """Filtruj 'marki' które są klasami betonu lub kodami: Pp4/0,6 itd."""
    if not brand:
        return True
    if re.search(r'[\d/,]', brand):
        return True
    if len(brand) < 2 or len(brand) > 40:
        return True
    return False

# ——— Prompt systemowy —————————————————————————————————————————
SYSTEM_PROMPT = """Jesteś ekspertem od materiałów budowlanych. Tworzysz treści SEO dla polskiego sklepu budowlanego MediaBud.
Odpowiadasz WYŁĄCZNIE poprawnym JSON bez markdown, bez ```json, bez komentarzy.

REGUŁY SEO NAME:
- Format: [Typ produktu] [Marka] [Model/Symbol] [Rozmiar/Pojemność/Ilość]
- Rozwijaj skróty: "Szt"→"szt.", "Kg"→"kg", "Cm"→"cm", "Mm"→"mm", "Ml"→"ml", "Kpl"→"kpl.", "Mb"→"mb", "Gr"→"grubość", "Dre"→"drewno", "Gk"→"GK", "Sil"→"silikon", "Akr"→"akryl"
- Max 80 znaków, pisownia: małe litery z wyjątkiem nazw własnych marek i skrótów technicznych (GK, PVC, EPS)
- Zachowaj WSZYSTKIE dane techniczne z oryginalnej nazwy (wymiary, kolor, typ, ilość)
- NIE skracaj — rozwijaj

REGUŁY OPISÓW:
- shortDescription: 1–2 zdania, max 250 znaków, czysto techniczne zastosowanie
- description: 3–5 zdań technicznych o właściwościach, składzie i zastosowaniu, min 400 znaków
- technicalSpec: array obiektów {"label":"...","value":"..."}, min 4 pozycje (Typ, Zastosowanie, Opakowanie/Wymiar, Marka lub Materiał)

ZAKAZ używania słów: gwarantuje, gwarantowana, idealny, idealny do, doskonały, najwyższej jakości, szeroko stosowany, innowacyjny, rewolucyjny, wyjątkowy, perfekcyjny, niezawodny, kompleksowy.
"""

def make_prompt(name: str, brand: str, cat: str) -> str:
    brand_line = f"Marka: {brand}" if brand else "Marka: nieznana"
    return (
        f"Produkt budowlany:\n"
        f"Nazwa oryginalna: {name}\n"
        f"{brand_line}\n"
        f"Kategoria: {cat}\n\n"
        f"Zwróć JSON:\n"
        f'{{"name":"<SEO nazwa max 80 znaków>","shortDescription":"<max 250 znaków>","description":"<min 400 znaków>","technicalSpec":[{{"label":"...","value":"..."}}]}}'
    )

# ——— Parser JSON ————————————————————————————————————————————
JSON_RE  = re.compile(r'\{[\s\S]*\}')
BAD_WORDS = re.compile(
    r'\b(gwarantuje|gwarantowana|idealny do|idealny|doskonały|najwyższej jakości|'
    r'szeroko stosowany|innowacyjny|rewolucyjny|wyjątkowy|perfekcyjny|niezawodny)\b',
    re.IGNORECASE
)

def clean_text(text: str) -> str:
    """Usuń zakazane słowa."""
    return BAD_WORDS.sub(lambda m: {
        'gwarantuje':         'zapewnia',
        'gwarantowana':       'zapewniana',
        'idealny do':         'przeznaczony do',
        'idealny':            'odpowiedni',
        'doskonały':          'dobry',
        'najwyższej jakości': 'wysokiej jakości',
        'szeroko stosowany':  'stosowany',
        'innowacyjny':        'nowoczesny',
        'rewolucyjny':        'zaawansowany',
        'wyjątkowy':          'charakterystyczny',
        'perfekcyjny':        'precyzyjny',
        'niezawodny':         'stabilny',
    }.get(m.group(0).lower(), m.group(0)), text)

def parse_result(raw: str, original_name: str) -> dict | None:
    if not raw:
        return None
    m = JSON_RE.search(raw)
    if not m:
        return None
    try:
        data = json.loads(m.group(0))
    except Exception:
        return None

    name_out = data.get("name", "").strip()
    short    = data.get("shortDescription", "").strip()
    desc     = data.get("description", "").strip()
    specs    = data.get("technicalSpec", [])

    # Walidacje
    if not name_out or len(name_out) < 6 or is_bad_name(name_out):
        name_out = original_name   # fallback — zostaje oryginał

    if not short or len(short) < 20:
        return None
    if not desc or len(desc) < 100:
        return None
    if not isinstance(specs, list) or len(specs) < 1:
        return None

    # Oczyść zakazane słowa
    name_out = clean_text(name_out)[:80]
    short    = clean_text(short)[:300]
    desc     = clean_text(desc)
    specs    = [{"label": s.get("label",""), "value": s.get("value","")}
                for s in specs if isinstance(s, dict) and s.get("label") and s.get("value")][:10]

    return {"name": name_out, "shortDescription": short, "description": desc, "technicalSpec": specs}

log("=== PIPELINE V4 ZAŁADOWANY — gotowy do uruchomienia ===")
log(f"START_OFFSET={START_OFFSET}, BATCH_SIZE={BATCH_SIZE}")
print("OK — kod przygotowany, uruchamiam run_pipeline_v4()")


[2026-05-29 15:55:14] === PIPELINE V4 ZAŁADOWANY — gotowy do uruchomienia ===


[2026-05-29 15:55:14] START_OFFSET=0, BATCH_SIZE=500


OK — kod przygotowany, uruchamiam run_pipeline_v4()


In [23]:

import sys, json, re, time, requests, asyncio
from datetime import datetime

sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

# ——— Config (ponowna definicja — background kernel) —————————
SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT_ID   = "nzcwegq7"
DATASET      = "production"
BATCH_SIZE   = 500
LLM_CHUNK    = 100
PATCH_CHUNK  = 50
LOG_FILE     = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log2.txt"
START_OFFSET = 0

def log(msg):
    ts   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

def groq(query, params=None):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    r   = requests.get(url,
                       headers={"Authorization": f"Bearer {SANITY_TOKEN}"},
                       params={"query": query, **(params or {})},
                       timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def sanity_patch(mutations):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/mutate/{DATASET}"
    r   = requests.post(url,
                        headers={"Authorization": f"Bearer {SANITY_TOKEN}",
                                 "Content-Type": "application/json"},
                        json={"mutations": mutations},
                        timeout=60)
    r.raise_for_status()
    return r.json()

BAD_NAME_RE = re.compile(
    r'^P-\d+$'
    r'|^[A-Z0-9\-]{1,8}$'
    r'|^[A-Z]{4,}$',
    re.IGNORECASE
)

def is_bad_name(name):
    if not name or len(name) < 4:
        return True
    caps = sum(1 for c in name if c.isupper())
    if caps / max(len(name), 1) > 0.7:
        return True
    if ' ' not in name and len(name) < 10:
        return True
    return bool(BAD_NAME_RE.match(name.strip()))

def is_bad_brand(brand):
    if not brand:
        return True
    if re.search(r'[\d/,]', brand):
        return True
    if len(brand) < 2 or len(brand) > 40:
        return True
    return False

SYSTEM_PROMPT = """Jesteś ekspertem od materiałów budowlanych. Tworzysz treści SEO dla polskiego sklepu budowlanego MediaBud.
Odpowiadasz WYŁĄCZNIE poprawnym JSON bez markdown, bez ```json, bez komentarzy.

REGUŁY SEO NAME:
- Format: [Typ produktu] [Marka] [Model/Symbol] [Rozmiar/Pojemność/Ilość]
- Rozwijaj skróty: Szt→szt., Kg→kg, Cm→cm, Mm→mm, Ml→ml, Kpl→kpl., Mb→mb, Gr→grubość, Dre→do drewna, Gk→GK, Sil→silikonowy, Akr→akrylowy, Wewn→wewnętrzny, Zewn→zewnętrzny, Elaw→elewacyjny, Wys→wysokość, Dl→długość, Szer→szerokość
- Max 80 znaków, małe litery z wyjątkiem: nazw marek, skrótów technicznych (GK, PVC, EPS, XPS, OSB)
- Zachowaj WSZYSTKIE dane techniczne (wymiary, kolor, typ, ilość, symbol modelu)
- NIE skracaj — rozwijaj pełne słowa

REGUŁY OPISÓW:
- shortDescription: 1-2 zdania techniczne, max 250 znaków, zastosowanie i główne właściwości
- description: 3-5 zdań technicznych, min 400 znaków, właściwości i zastosowanie — bez marketingu
- technicalSpec: min 4 obiekty {"label":"...","value":"..."}: Typ, Zastosowanie, Opakowanie lub Wymiar, Marka lub Materiał

ZAKAZ używania: gwarantuje, idealny, doskonały, najwyższej jakości, szeroko stosowany, innowacyjny, rewolucyjny, wyjątkowy, perfekcyjny, niezawodny."""

def make_prompt(name, brand, cat):
    brand_line = f"Marka: {brand}" if brand else "Marka: nieznana"
    return (
        f"Produkt budowlany:\nNazwa oryginalna: {name}\n{brand_line}\nKategoria: {cat}\n\n"
        f'Zwróć JSON: {{"name":"<SEO nazwa max 80 znaków>","shortDescription":"<max 250 znaków>",'
        f'"description":"<min 400 znaków>","technicalSpec":[{{"label":"...","value":"..."}}]}}'
    )

JSON_RE   = re.compile(r'\{[\s\S]*\}')
BAD_WORDS = re.compile(
    r'\b(gwarantuje|gwarantowana|idealny do|idealny|doskonały|doskonała|'
    r'najwyższej jakości|szeroko stosowany|innowacyjny|rewolucyjny|'
    r'wyjątkowy|perfekcyjny|niezawodny)\b', re.IGNORECASE
)
REPLACE_MAP = {
    'gwarantuje':'zapewnia','gwarantowana':'zapewniana',
    'idealny do':'przeznaczony do','idealny':'odpowiedni',
    'doskonały':'dobry','doskonała':'dobra',
    'najwyższej jakości':'wysokiej jakości',
    'szeroko stosowany':'stosowany',
    'innowacyjny':'nowoczesny','rewolucyjny':'zaawansowany',
    'wyjątkowy':'charakterystyczny','perfekcyjny':'precyzyjny',
    'niezawodny':'stabilny',
}

def clean_text(text):
    return BAD_WORDS.sub(lambda m: REPLACE_MAP.get(m.group(0).lower(), m.group(0)), text)

def parse_result(raw, original_name):
    if not raw:
        return None
    m = JSON_RE.search(raw)
    if not m:
        return None
    try:
        data = json.loads(m.group(0))
    except Exception:
        return None

    name_out = data.get("name","").strip()
    short    = data.get("shortDescription","").strip()
    desc     = data.get("description","").strip()
    specs    = data.get("technicalSpec",[])

    if not name_out or len(name_out) < 6 or is_bad_name(name_out):
        name_out = original_name

    if not short or len(short) < 20:
        return None
    if not desc or len(desc) < 100:
        return None
    if not isinstance(specs, list) or len(specs) < 1:
        return None

    name_out = clean_text(name_out)[:80]
    short    = clean_text(short)[:300]
    desc     = clean_text(desc)
    specs    = [{"label": s.get("label",""), "value": s.get("value","")}
                for s in specs
                if isinstance(s, dict) and s.get("label") and s.get("value")][:10]

    return {"name": name_out, "shortDescription": short,
            "description": desc, "technicalSpec": specs}

# ——— Główna pętla ——————————————————————————————————————————————
async def run_pipeline_v4():
    log("=== PIPELINE V4 START ===")
    offset       = START_OFFSET
    total_updated = 0
    total_skip    = 0
    total_err     = 0

    while True:
        # 1. Pobierz batch z Sanity
        q = (
            f'*[_type == "product"] | order(_id asc) [{offset}...{offset+BATCH_SIZE}]'
            f'{{_id, name, "brandName": brand->name, "catName": category->name}}'
        )
        try:
            products = groq(q)
        except Exception as e:
            log(f"ERROR groq offset {offset}: {e}")
            break

        if not products:
            log(f"Koniec danych na offset {offset}")
            break

        log(f"Offset {offset}: pobrano {len(products)} produktów")

        # 2. Filtruj
        good = []
        for p in products:
            name  = (p.get("name") or "").strip()
            if is_bad_name(name):
                total_skip += 1
                continue
            brand = p.get("brandName","") or ""
            if is_bad_brand(brand):
                brand = ""
            cat   = p.get("catName","") or "materiały budowlane"
            good.append({"_id": p["_id"], "name": name, "brand": brand, "cat": cat})

        log(f"  Po filtracji: {len(good)} dobrych, {len(products)-len(good)} pominiętych")

        if not good:
            offset += BATCH_SIZE
            continue

        # 3. Generuj opisy przez batch_llm w kawałkach po LLM_CHUNK
        results_map = {}
        for i in range(0, len(good), LLM_CHUNK):
            chunk = good[i:i+LLM_CHUNK]
            prompts = [make_prompt(p["name"], p["brand"], p["cat"]) for p in chunk]
            try:
                raw_list = await batch_llm(prompts, system=SYSTEM_PROMPT)
            except Exception as e:
                log(f"  batch_llm error chunk {i}: {e}")
                raw_list = [None] * len(chunk)

            for prod, raw in zip(chunk, raw_list):
                parsed = parse_result(raw, prod["name"])
                if parsed:
                    results_map[prod["_id"]] = parsed

        parsed_count = len(results_map)
        log(f"  LLM sparsowanych: {parsed_count} / {len(good)}")

        # 4. PATCH do Sanity
        ids_to_patch = list(results_map.keys())
        patched = 0
        for j in range(0, len(ids_to_patch), PATCH_CHUNK):
            chunk_ids = ids_to_patch[j:j+PATCH_CHUNK]
            mutations = []
            for doc_id in chunk_ids:
                r = results_map[doc_id]
                mutations.append({
                    "patch": {
                        "id": doc_id,
                        "set": {
                            "name":             r["name"],
                            "shortDescription": r["shortDescription"],
                            "description":      r["description"],
                            "technicalSpec":    r["technicalSpec"],
                        }
                    }
                })
            try:
                sanity_patch(mutations)
                patched += len(chunk_ids)
            except Exception as e:
                log(f"  PATCH error: {e}")
                total_err += len(chunk_ids)

        total_updated += patched
        total_err     += (parsed_count - patched)
        total_skip    += (len(good) - parsed_count)

        log(f"  PATCH: {patched} dokumentów zapisanych")
        log(f"  Postęp: updated={total_updated}, skip={total_skip}, err={total_err}")

        offset += BATCH_SIZE
        time.sleep(0.5)   # chwila przerwy między transzami

    log(f"=== PIPELINE V4 KONIEC === updated={total_updated}, skip={total_skip}, err={total_err}")
    return total_updated, total_skip, total_err

# Uruchom
result = asyncio.run(run_pipeline_v4())
print(f"WYNIK KOŃCOWY: updated={result[0]}, skip={result[1]}, err={result[2]}")


RuntimeError: asyncio.run() cannot be called from a running event loop

In [27]:

import sys, json, re, time, requests, asyncio
from datetime import datetime

sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

SANITY_TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT_ID   = "nzcwegq7"
DATASET      = "production"
BATCH_SIZE   = 500
LLM_CHUNK    = 100
PATCH_CHUNK  = 50
LOG_FILE     = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log2.txt"
START_OFFSET = 0

def log(msg):
    ts   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

def groq(query, params=None):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/query/{DATASET}"
    r   = requests.get(url, headers={"Authorization": f"Bearer {SANITY_TOKEN}"},
                       params={"query": query, **(params or {})}, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def sanity_patch(mutations):
    url = f"https://{PROJECT_ID}.api.sanity.io/v2021-10-21/data/mutate/{DATASET}"
    r   = requests.post(url,
                        headers={"Authorization": f"Bearer {SANITY_TOKEN}",
                                 "Content-Type": "application/json"},
                        json={"mutations": mutations}, timeout=60)
    r.raise_for_status()
    return r.json()

BAD_NAME_RE = re.compile(r'^P-\d+$|^[A-Z0-9\-]{1,8}$|^[A-Z]{4,}$', re.IGNORECASE)

def is_bad_name(name):
    if not name or len(name) < 4: return True
    caps = sum(1 for c in name if c.isupper())
    if caps / max(len(name),1) > 0.7: return True
    if ' ' not in name and len(name) < 10: return True
    return bool(BAD_NAME_RE.match(name.strip()))

def is_bad_brand(brand):
    if not brand: return True
    if re.search(r'[\d/,]', brand): return True
    if len(brand) < 2 or len(brand) > 40: return True
    return False

SYSTEM_PROMPT = """Jesteś ekspertem od materiałów budowlanych. Tworzysz treści SEO dla polskiego sklepu budowlanego MediaBud.
Odpowiadasz WYŁĄCZNIE poprawnym JSON bez markdown, bez ```json, bez komentarzy.

REGUŁY SEO NAME:
- Format: [Typ produktu] [Marka] [Model/Symbol] [Rozmiar/Pojemność/Ilość]
- Rozwijaj skróty: Szt→szt., Kg→kg, Cm→cm, Mm→mm, Ml→ml, Kpl→kpl., Mb→mb, Dre→do drewna, Gk→GK, Wewn→wewnętrzny, Zewn→zewnętrzny, Elaw→elewacyjny, Bial→biały, Szar→szary
- Max 80 znaków, małe litery z wyjątkiem: nazw marek, skrótów technicznych (GK, PVC, EPS, XPS, OSB)
- Zachowaj WSZYSTKIE dane techniczne (wymiary, kolor, typ, ilość, model)
- NIE skracaj — rozwijaj pełne słowa

REGUŁY OPISÓW:
- shortDescription: 1-2 zdania techniczne, max 250 znaków
- description: 3-5 zdań technicznych, min 400 znaków, bez marketingu
- technicalSpec: min 4 obiekty {"label":"...","value":"..."}: Typ, Zastosowanie, Opakowanie lub Wymiar, Marka lub Materiał

ZAKAZ: gwarantuje, idealny, doskonały, najwyższej jakości, szeroko stosowany, innowacyjny, wyjątkowy, perfekcyjny, niezawodny."""

def make_prompt(name, brand, cat):
    bl = f"Marka: {brand}" if brand else "Marka: nieznana"
    return (f"Produkt budowlany:\nNazwa oryginalna: {name}\n{bl}\nKategoria: {cat}\n\n"
            f'Zwróć JSON: {{"name":"<SEO max 80 znaków>","shortDescription":"<max 250 znaków>",'
            f'"description":"<min 400 znaków>","technicalSpec":[{{"label":"...","value":"..."}}]}}')

JSON_RE   = re.compile(r'\{[\s\S]*\}')
BAD_WORDS = re.compile(
    r'\b(gwarantuje|gwarantowana|idealny do|idealny|doskonały|doskonała|'
    r'najwyższej jakości|szeroko stosowany|innowacyjny|rewolucyjny|'
    r'wyjątkowy|perfekcyjny|niezawodny)\b', re.IGNORECASE)
REPLACE_MAP = {
    'gwarantuje':'zapewnia','gwarantowana':'zapewniana',
    'idealny do':'przeznaczony do','idealny':'odpowiedni',
    'doskonały':'dobry','doskonała':'dobra',
    'najwyższej jakości':'wysokiej jakości','szeroko stosowany':'stosowany',
    'innowacyjny':'nowoczesny','rewolucyjny':'zaawansowany',
    'wyjątkowy':'charakterystyczny','perfekcyjny':'precyzyjny','niezawodny':'stabilny',
}

def clean_text(t):
    return BAD_WORDS.sub(lambda m: REPLACE_MAP.get(m.group(0).lower(), m.group(0)), t)

def parse_result(raw, orig):
    if not raw: return None
    m = JSON_RE.search(raw)
    if not m: return None
    try: data = json.loads(m.group(0))
    except: return None
    name_out = data.get("name","").strip()
    short    = data.get("shortDescription","").strip()
    desc     = data.get("description","").strip()
    specs    = data.get("technicalSpec",[])
    if not name_out or len(name_out)<6 or is_bad_name(name_out): name_out = orig
    if not short or len(short)<20: return None
    if not desc or len(desc)<100: return None
    if not isinstance(specs,list) or len(specs)<1: return None
    name_out = clean_text(name_out)[:80]
    short    = clean_text(short)[:300]
    desc     = clean_text(desc)
    specs    = [{"label":s.get("label",""),"value":s.get("value","")}
                for s in specs if isinstance(s,dict) and s.get("label") and s.get("value")][:10]
    return {"name":name_out,"shortDescription":short,"description":desc,"technicalSpec":specs}

async def run_pipeline_v4():
    log("=== PIPELINE V4 START ===")
    offset, total_updated, total_skip, total_err = START_OFFSET, 0, 0, 0
    while True:
        q = (f'*[_type == "product"] | order(_id asc) [{offset}...{offset+BATCH_SIZE}]'
             f'{{_id, name, "brandName": brand->name, "catName": category->name}}')
        try:
            products = groq(q)
        except Exception as e:
            log(f"ERROR groq offset {offset}: {e}"); break
        if not products:
            log(f"Koniec danych na offset {offset}"); break

        log(f"Offset {offset}: pobrano {len(products)} produktów")
        good = []
        for p in products:
            name = (p.get("name") or "").strip()
            if is_bad_name(name): total_skip += 1; continue
            brand = p.get("brandName","") or ""
            if is_bad_brand(brand): brand = ""
            cat   = p.get("catName","") or "materiały budowlane"
            good.append({"_id":p["_id"],"name":name,"brand":brand,"cat":cat})
        log(f"  Po filtracji: {len(good)} dobrych, {len(products)-len(good)} pominiętych")

        results_map = {}
        for i in range(0, len(good), LLM_CHUNK):
            chunk   = good[i:i+LLM_CHUNK]
            prompts = [make_prompt(p["name"],p["brand"],p["cat"]) for p in chunk]
            try:
                raw_list = await batch_llm(prompts, system=SYSTEM_PROMPT)
            except Exception as e:
                log(f"  batch_llm error chunk {i}: {e}"); raw_list = [None]*len(chunk)
            for prod, raw in zip(chunk, raw_list):
                parsed = parse_result(raw, prod["name"])
                if parsed: results_map[prod["_id"]] = parsed

        log(f"  LLM sparsowanych: {len(results_map)} / {len(good)}")
        ids_to_patch = list(results_map.keys())
        patched = 0
        for j in range(0, len(ids_to_patch), PATCH_CHUNK):
            chunk_ids = ids_to_patch[j:j+PATCH_CHUNK]
            mutations = [{"patch":{"id":did,"set":{
                "name":             results_map[did]["name"],
                "shortDescription": results_map[did]["shortDescription"],
                "description":      results_map[did]["description"],
                "technicalSpec":    results_map[did]["technicalSpec"],
            }}} for did in chunk_ids]
            try:
                sanity_patch(mutations); patched += len(chunk_ids)
            except Exception as e:
                log(f"  PATCH error: {e}"); total_err += len(chunk_ids)
        total_updated += patched
        total_err     += (len(results_map) - patched)
        total_skip    += (len(good) - len(results_map))
        log(f"  PATCH: {patched} zapisanych | Postęp: updated={total_updated}, skip={total_skip}, err={total_err}")
        offset += BATCH_SIZE
        time.sleep(0.3)

    log(f"=== PIPELINE V4 KONIEC === updated={total_updated}, skip={total_skip}, err={total_err}")
    return total_updated, total_skip, total_err

# POPRAWKA: w Jupyter używamy await bezpośrednio
result = await run_pipeline_v4()
print(f"WYNIK: updated={result[0]}, skip={result[1]}, err={result[2]}")


[2026-05-29 16:24:34] === PIPELINE V4 START ===


[2026-05-29 16:24:34] Offset 0: pobrano 500 produktów


[2026-05-29 16:24:34]   Po filtracji: 435 dobrych, 65 pominiętych


Batch complete: 100/100 succeeded


Batch complete: 100/100 succeeded


...

Batch complete: 45/45 succeeded
[2026-05-29 18:36:48]   LLM sparsowanych: 245 / 245


[2026-05-29 18:36:56]   PATCH: 245 zapisanych | Postęp: updated=13020, skip=2901, err=0


[2026-05-29 18:36:56] Koniec danych na offset 16000


[2026-05-29 18:36:56] === PIPELINE V4 KONIEC === updated=13020, skip=2901, err=0


WYNIK: updated=13020, skip=2901, err=0


In [3]:

import urllib.request, urllib.parse, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/query/production"

def sanity_query(groq: str):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# 4 liczniki jednym zapytaniem przez array
stats_groq = '''{
  "total": count(*[_type=="product"]),
  "with_name": count(*[_type=="product" && defined(name) && name != ""]),
  "with_short": count(*[_type=="product" && defined(shortDescription) && shortDescription != ""]),
  "with_desc": count(*[_type=="product" && defined(description) && description != ""])
}'''

stats = sanity_query(stats_groq)
print("=== STATYSTYKI ===")
for k, v in stats.items():
    print(f"  {k}: {v}")


=== STATYSTYKI ===
  total: 15921
  with_desc: 13250
  with_name: 15921
  with_short: 13380


In [7]:

import urllib.request, urllib.parse, json, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/query/production"

def sanity_query(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

sample = sanity_query('*[_type=="product" && defined(name) && defined(shortDescription)]{_id,name,shortDescription,brand}[0...50]')

BANNED = ["gwarantuje", "idealny", "idealna", "idealne", "doskonały", "doskonała", "doskonałe",
          "najwyższej jakości", "innowacyjny", "innowacyjna", "innowacyjne", "najlepszy", "najlepsza"]
SKU_RE = re.compile(r'^[A-Z0-9\-]{4,20}$')
MARKETING_START = re.compile(r'^(Odkryj|Poznaj|Poczuj|Doświadcz|Zadbaj|Zapewnij|Ciesz się|Skorzystaj)', re.IGNORECASE)

bad_banned = []
bad_long   = []
bad_sku    = []
bad_short  = []

for p in sample:
    name  = (p.get("name") or "").strip()
    short = (p.get("shortDescription") or "").strip()

    # 1. Zakazane słowa
    for w in BANNED:
        if w.lower() in name.lower():
            bad_banned.append({"id": p["_id"], "name": name, "word": w})
            break

    # 2. Długość
    if len(name) > 80:
        bad_long.append({"id": p["_id"], "name": name, "len": len(name)})

    # 3. Samo SKU
    if SKU_RE.match(name):
        bad_sku.append({"id": p["_id"], "name": name})

    # 4. shortDesc zaczyna od marketingowej frazy
    if MARKETING_START.match(short):
        bad_short.append({"id": p["_id"], "short": short[:80]})

print(f"Próbka: {len(sample)} produktów")
print(f"\n--- Zakazane słowa w name: {len(bad_banned)} ---")
for x in bad_banned[:5]:
    print(f"  [{x['word']}] {x['name'][:70]}")

print(f"\n--- Name > 80 znaków: {len(bad_long)} ---")
for x in bad_long[:5]:
    print(f"  ({x['len']}z) {x['name'][:80]}")

print(f"\n--- Samo SKU: {len(bad_sku)} ---")
for x in bad_sku[:5]:
    print(f"  {x['name']}")

print(f"\n--- shortDescription zaczyna od frazy marketingowej: {len(bad_short)} ---")
for x in bad_short[:5]:
    print(f"  {x['short']}")

print(f"\n=== PODSUMOWANIE JAKOŚCI (próbka 50) ===")
print(f"  bad_banned  : {len(bad_banned)}")
print(f"  bad_long    : {len(bad_long)}")
print(f"  bad_sku     : {len(bad_sku)}")
print(f"  bad_short   : {len(bad_short)}")


Próbka: 50 produktów

--- Zakazane słowa w name: 0 ---

--- Name > 80 znaków: 0 ---

--- Samo SKU: 5 ---
  P-0000235
  P-0000757
  P-0000976
  P-0001110
  P-0001218

--- shortDescription zaczyna od frazy marketingowej: 0 ---

=== PODSUMOWANIE JAKOŚCI (próbka 50) ===
  bad_banned  : 0
  bad_long    : 0
  bad_sku     : 5
  bad_short   : 0


In [11]:

import urllib.request, urllib.parse, json, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/query/production"

def sanity_query(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# SKU-only: name pasuje do wzorca P-XXXXXXX
sku_count = sanity_query('count(*[_type=="product" && defined(name) && name matches "P-*"])')
print(f"Produkty z name=SKU (P-*): {sku_count}")

# Produkty z opisem i nazwą SEO (nie SKU)
good_count = sanity_query('count(*[_type=="product" && defined(name) && !(name matches "P-*") && defined(shortDescription) && shortDescription != ""])')
print(f"Produkty z prawdziwą nazwą SEO + shortDesc: {good_count}")

# Bez żadnego opisu
no_desc = sanity_query('count(*[_type=="product" && (!defined(description) || description == "") && (!defined(shortDescription) || shortDescription == "")])')
print(f"Produkty bez żadnego opisu: {no_desc}")

print(f"\n=== KOMPLETNE STATYSTYKI ===")
total = 15921
with_name = 15921
with_short = 13380
with_desc = 13250
sku_only = sku_count
good = good_count
print(f"  Total produktów:         {total}")
print(f"  Z defined(name):         {with_name} ({with_name/total*100:.1f}%)")
print(f"  Z shortDescription:      {with_short} ({with_short/total*100:.1f}%)")
print(f"  Z description:           {with_desc} ({with_desc/total*100:.1f}%)")
print(f"  name = SKU (P-*):        {sku_only} ({sku_only/total*100:.1f}%)")
print(f"  Pełna jakość (SEO+short): {good} ({good/total*100:.1f}%)")
print(f"  Bez opisów:              {no_desc} ({no_desc/total*100:.1f}%)")


HTTPError: HTTP Error 400: Bad Request

In [15]:

import urllib.request, urllib.parse, json, re

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/query/production"

def sanity_query(groq):
    url = BASE + "?query=" + urllib.parse.quote(groq)
    req = urllib.request.Request(url, headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]

# SKU-only: name zaczyna się od "P-"
sku_count = sanity_query('count(*[_type=="product" && defined(name) && string::startsWith(name, "P-")])')
print(f"Produkty z name=SKU (P-*): {sku_count}")

# Pełna jakość: prawdziwa nazwa SEO + shortDesc
good_count = sanity_query('count(*[_type=="product" && defined(name) && !string::startsWith(name, "P-") && defined(shortDescription) && shortDescription != ""])')
print(f"Produkty z SEO name + shortDesc: {good_count}")

# Bez żadnego opisu
no_desc = sanity_query('count(*[_type=="product" && (!defined(description) || description == "") && (!defined(shortDescription) || shortDescription == "")])')
print(f"Produkty bez żadnego opisu: {no_desc}")

total = 15921
with_name = 15921
with_short = 13380
with_desc = 13250

print(f"\n=== KOMPLETNE STATYSTYKI ===")
print(f"  Total:                    {total}")
print(f"  z defined(name):          {with_name} ({with_name/total*100:.1f}%)")
print(f"  z shortDescription:       {with_short} ({with_short/total*100:.1f}%)")
print(f"  z description:            {with_desc} ({with_desc/total*100:.1f}%)")
print(f"  name = SKU (P-*):         {sku_count} ({sku_count/total*100:.1f}%)")
print(f"  Pełna jakość SEO:         {good_count} ({good_count/total*100:.1f}%)")
print(f"  Bez opisów:               {no_desc} ({no_desc/total*100:.1f}%)")


Produkty z name=SKU (P-*): 2614
Produkty z SEO name + shortDesc: 13229


Produkty bez żadnego opisu: 2541

=== KOMPLETNE STATYSTYKI ===
  Total:                    15921
  z defined(name):          15921 (100.0%)
  z shortDescription:       13380 (84.0%)
  z description:            13250 (83.2%)
  name = SKU (P-*):         2614 (16.4%)
  Pełna jakość SEO:         13229 (83.1%)
  Bez opisów:               2541 (16.0%)
